To Do:

- what attributes are relevant for the user in total?
- which can bestored on the Dataset Level?
- which can be stored on the DataArray Level?
- write tests for the storrage process

## User Relevant Attributes

- what noise is underlying?
- what trend is underlying?
- how many shifts are in the PONDS
- for each shift:
    - what kind is it? -> to select matching detector
    - which parameters were used to generate it? -> reproducability
    - if not single:
        - which blobgenerator was used?
        - which parameters were used?   -> reproducability

## Distributing onto Levels

| Dataset Level         | DataArray Level     |
|-----------------------|---------------------|
| - noise               | - type of shift     |
| - trend               | - shift parameters  |
| - number of shifts    | - type of blob      |
| - type of shift       | - blob parameters   |

## Try out Attribute Addition

In [4]:
import numpy as np
import xarray as xr

# Create sample data
dims = ('time', 'x', 'y')
shape = (10, 3, 2)
data1 = np.random.rand(*shape)
data2 = np.random.choice([True, False], size=shape)
data3 = np.random.choice([True, False], size=shape)

# Create DataArrays
da1 = xr.DataArray(data1, dims=dims, attrs={
    'noise': 0.01,
    'background': 0.005,
})
da2 = xr.DataArray(data2, dims=dims, attrs={
    'shift_type': 'SIGMOID',
    'shift_params': {
        "shift_steepness": 1,
        "shift_magnitude": 1,
        "sigma_time": 0,
    },
    'blob_type': 'GAUSS',
    'blob_params': {
        "sigma_x": 0.1,
        "sigma_y": 0.1,
    }
})
da3 = xr.DataArray(data3, dims=dims, attrs={
    'shift_type': 'LORENZ',
    'shift_params': {
        "a": 1,
        "b": 1,
    },
    'blob_type': 'GAUSS',
    'blob_params': {
        "sigma_x": 0.1,
        "sigma_y": 0.1,
    }
})

# Combine into a Dataset
ds = xr.Dataset({'ts': da1, 'c1': da2, 'c2': da3}, attrs={
    'noise': 0.01,
    'background': 0.005,
    'num_clusters': 2,
    'c1': 'SIGMOID',
    'c2': 'LORENZ'
})

ds

<xarray.Dataset> Size: 600B
Dimensions:  (time: 10, x: 3, y: 2)
Dimensions without coordinates: time, x, y
Data variables:
    ts       (time, x, y) float64 480B 0.5497 0.4353 0.4204 ... 0.8003 0.6018
    c1       (time, x, y) bool 60B True False False True ... False True True
    c2       (time, x, y) bool 60B True False False False ... False False True
Attributes:
    noise:         0.01
    background:    0.005
    num_clusters:  2
    c1:            SIGMOID
    c2:            LORENZ

In [5]:
ts = ds['ts']
c1 = ds['c1']
c2 = ds['c2']

c1

<xarray.DataArray 'c1' (time: 10, x: 3, y: 2)> Size: 60B
array([[[ True, False],
        [False,  True],
        [ True, False]],

       [[False, False],
        [False, False],
        [False,  True]],

       [[ True,  True],
        [ True, False],
        [False, False]],

       [[False,  True],
        [False,  True],
        [False, False]],

       [[False, False],
        [ True, False],
        [ True, False]],

       [[False,  True],
        [ True, False],
        [False,  True]],

       [[ True,  True],
        [ True, False],
        [ True,  True]],

       [[False, False],
        [ True,  True],
        [False, False]],

       [[False, False],
        [ True, False],
        [False, False]],

       [[ True,  True],
        [False, False],
        [ True,  True]]])
Dimensions without coordinates: time, x, y
Attributes:
    shift_type:    SIGMOID
    shift_params:  {'shift_steepness': 1, 'shift_magnitude': 1, 'sigma_time': 0}
    blob_type:     GAUSS
    blob_params:   {'sigma_x': 0.1, 'sigma_y': 0.1}

## Try Out in PONDS

In [1]:
from ponds import PONDS, GAUSS, SIGMOID
import matplotlib.pyplot as plt

In [2]:
background_noise = 0.05
background_trend = 0.002

pd = PONDS(
    time_size=200,
    background_noise=background_noise,
    background_trend=background_trend,
)


# adding a random shift to the time series
pd.add_shift(
    random_seed=2,
    shift_method=SIGMOID(),
)


# adding a random shift to the time series
pd.add_shift(
    random_seed=2,
    shift_method=SIGMOID(),
)

pd.data

<xarray.Dataset> Size: 4kB
Dimensions:  (time: 200, lat: 1, lon: 1)
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 8B -90.0
  * lon      (lon) float64 8B -180.0
Data variables:
    ts       (time, lat, lon) float64 2kB 0.02484 -0.004903 ... 2.419 2.36
    c1       (time, lat, lon) bool 200B False False False ... False False False
    c2       (time, lat, lon) bool 200B False False False ... False False False
Attributes:
    background_trend:  0.002
    background_noise:  0.05
    num_clusters:      2
    c1:                SIGMOID
    c2:                SIGMOID

In [3]:
ts = pd.data['ts']
c1 = pd.data['c1']
c2 = pd.data['c2']

c1

<xarray.DataArray 'c1' (time: 200, lat: 1, lon: 1)> Size: 200B
array([[[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

...

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]],

       [[False]]])
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 8B -90.0
  * lon      (lon) float64 8B -180.0
Attributes:
    shift_type:    SIGMOID
    shift_params:  {'shift_steepness': 2.8160596078919333, 'shift_magnitude':...
    blob_type:     SINGLE
    blob_params:   {}

In [4]:
# initialize PONDS object
pd = PONDS(
    lat_size=20,
    lon_size=40,
    time_size=200,
    background_noise=0.05,
    background_trend=0.0002,
)

# add random shift
pd.add_shift(
    random_seed=1,
    shape_method=GAUSS(random_seed=1),
    shift_method=SIGMOID(),
)

# add shift with custom parameters
shift_dict = {
    "shift_steepness": 0.2,
    "shift_magnitude": 0.5,
}
pd.add_shift(
    random_seed=4,
    shape_method=GAUSS(random_seed=4),
    shift_method=SIGMOID(**shift_dict),
)


pd.data

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 200, lat: 20, lon: 40)
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Data variables:
    ts       (time, lat, lon) float64 1MB 0.02484 0.01789 ... 0.056 0.07663
    c1       (time, lat, lon) bool 160kB False False False ... False False False
    c2       (time, lat, lon) bool 160kB False False False ... False False False
Attributes:
    background_trend:  0.0002
    background_noise:  0.05
    num_clusters:      2
    c1:                SIGMOID
    c2:                SIGMOID

In [5]:
ts = pd.data['ts']
c1 = pd.data['c1']
c2 = pd.data['c2']

c1

<xarray.DataArray 'c1' (time: 200, lat: 20, lon: 40)> Size: 160kB
array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
...
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]]])
Coordinates:
  * time     (time) float64 2kB 0.0 1.005 2.01 3.015 ... 197.0 198.0 199.0 200.0
  * lat      (lat) float64 160B -90.0 -80.53 -71.05 -61.58 ... 71.05 80.53 90.0
  * lon      (lon) float64 320B -180.0 -170.8 -161.5 ... 161.5 170.8 180.0
Attributes:
    shift_type:    SIGMOID
    shift_params:  {'shift_steepness': 1.7201338362256695, 'shift_magnitude':...
    blob_type:     GAUSS
    blob_params:   {'center_lat': -13.276479247588156, 'center_lon': 79.31681...